# 02 — Alex's Morning Scan

**Notebook 2 of the *Developer Guide to Disciplined Trading* series.**

> Prerequisite: [`01-foundations-techtrade-and-analysis.ipynb`](./01-foundations-techtrade-and-analysis.ipynb) — you should already have a working `.venv_win`, a configured `fmp_cached_api_key`, and have seen `obb.techtrade.about()` return cleanly.

---

## Learning objectives

By working through this notebook you'll:

1. Understand what a **systematic morning scan** is and why it beats discretionary "check the news and pick something" trading. Alexander Elder's [*Trading for a Living*](https://en.wikipedia.org/wiki/Alexander_Elder) (Wiley, 1993) codifies this as the three pillars: **method, money management, mind** — a written system, sized correctly, executed without emotion.
2. See how [**GICS sectors**](https://www.investopedia.com/terms/g/gics.asp) partition the equity market and why sector-level analysis matters more than individual-stock hunting.
3. Learn what a [**confluence signal**](https://www.investopedia.com/terms/c/confluence.asp) is — multiple independent indicators agreeing on direction — and why that's mathematically stronger than any single indicator.
4. Apply the [**1% risk rule**](https://www.investopedia.com/articles/trading/09/risk-management.asp) (popularized by Van Tharp in [*Trade Your Way to Financial Freedom*](https://vantharp.com/trade-your-way-to-financial-freedom/), McGraw-Hill 1998) via `obb.techtrade.scan(risk=0.01)`.
5. Produce a repeatable morning artifact: the 6-sheet Excel workbook that becomes the day's trading journal.

## Recap: who's Alex?

Alex is the software engineer from notebook 01: literate in Python, day-trades on the side, has been losing more than he likes by trading on hunches, and now wants a system. Notebook 01 showed him the breadth of `obb.techtrade.*`. This notebook is the **first deep workflow**: his morning routine.

## What is a "trade plan"?

A trade plan is a **complete pre-committed specification** of a single trade: entry price, stop-loss, target, position size, holding rules, and — crucially — the **reasoning** for why this trade should work. Discretionary traders decide these values *as the trade happens* (or worse, after entering); systematic traders write the plan first, then either follow it or don't take the trade. The value of writing it down is not the plan itself — it's the **discipline of not deviating from it once real money is on the line**. See [Investopedia: Trading Plan](https://www.investopedia.com/terms/t/trading-plan.asp) for the standard components list.

Every plan techtrade produces is signed by the confluence engine so you can inspect *which* indicators voted long/short with *what* weight. §7 below walks through the audit trail.

## What this notebook covers

By the end, Alex has produced his **morning trade-plan workbook** — a 6-sheet Excel file containing the day's top setups, ranked by conviction, with the audit trail (which indicators voted long/short on each ticker, what entry/stop/target prices, what position size at 1% account risk). This is the artifact he can stare at over coffee before the market opens.

**Mapped to the README's command surface (§ Commands):**

| Step | Command | What it gives Alex |
|---|---|---|
| 1 | `obb.techtrade.segments()` | The 11 GICS sectors he could scan today |
| 2 | `obb.techtrade.movers(segment=...)` | The top movers within each sector (sanity check) |
| 3 | `obb.techtrade.scan(metric=..., top_n=..., preset=..., risk=...)` | **The headline call** — ranked actionable plans across all 11 sectors |
| 4 | `obb.techtrade.export(plans=...)` | The printable Excel workbook |

**Wall-clock budget:** ~3-8 minutes if your `fmp_cached` cache is warm; up to ~20 minutes cold (Excel export is the slow step). All cells call real `fmp_cached`; nothing is faked.

## What this notebook is NOT

- **Not a backtest.** Robustness gating lives in notebook 04 (`obb.techtrade.validate`). If the term is new, see [Investopedia: Backtesting](https://www.investopedia.com/terms/b/backtesting.asp) — the practice of applying a strategy's rules to historical data to estimate how it would have performed.
- **Not a single-position deep dive.** Notebook 03 takes one ticker from this scan and shows the full plan → orders → paper-fill chain.
- **Not investment advice.** Every plan this notebook produces is *research / paper-trading material*. Alex pulls the trigger; the engine never does. Before risking real capital, read the SEC's beginner guidance at [investor.gov](https://www.investor.gov/introduction-investing/investing-basics/save-and-invest) and the FINRA [broker checklist](https://www.finra.org/investors/learn-to-invest).

---

*Reading time: ~30 minutes if you follow the linked concepts. Coding time: 5-15 minutes for the whole notebook.*

## 1. Setup probe

First a 5-second sanity check that the environment is the same one notebook 01 set up. If anything fails, fix it per `01-foundations` §1.

In [6]:
import importlib.util
import sys
from pathlib import Path

REQUIRED = ["openbb", "openbb_techtrade", "openbb_fmp_cached", "openpyxl", "pandas_ta_classic"]
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    raise RuntimeError(f"Missing required packages: {missing}. See 01-foundations §1.")

settings = Path.home() / ".openbb_platform" / "user_settings.json"
if not settings.exists():
    raise RuntimeError(f"Missing credentials file: {settings}")

print(f"Python {sys.version.split()[0]} | {len(REQUIRED)} required packages present | credentials configured")

Python 3.12.13 | 5 required packages present | credentials configured


In [7]:
# First obb import is ~5-10s (openbb.build()); later cells are instant.
from openbb import obb
from datetime import date

print(f"obb loaded; today is {date.today().isoformat()}")
print(f"techtrade version: {obb.techtrade.about().results.get('extension_version', '?')}")

obb loaded; today is 2026-07-08
techtrade version: 0.1.0


## 2. The sector universe

### What is a GICS sector?

The [**Global Industry Classification Standard (GICS)**](https://www.investopedia.com/terms/g/gics.asp) is the taxonomy MSCI and S&P Dow Jones use to slice the equity market into 11 top-level sectors (Financials, Health Care, Information Technology, Energy, etc.). Every public equity is classified into **exactly one** sector, which means sector-level analysis gives a mutually-exclusive, collectively-exhaustive view of the market. That property is why professional risk models and factor-based strategies almost always start with sector exposures.

### Why scan sector-by-sector instead of the whole market?

Two reasons:

1. **[Sector rotation](https://www.investopedia.com/terms/s/sector-rotation.asp)** — Money moves between sectors on macro cycles (defensives lead in recessions; cyclicals lead in early recovery). A stock isn't just a stock; it inherits its sector's regime. Scanning per sector surfaces movers *relative to their peers*, not swamped by whatever the whole market is doing that day.
2. **Diversification arithmetic** — Correlations *within* a sector are high (Apple and Microsoft move together most days); correlations *across* sectors are lower. If Alex's top 5 picks all come from Info Tech, he's effectively taken one bet 5 times. Sector-aware scanning + a per-sector cap (see §6) enforces genuine diversification — the same principle Harry Markowitz formalized in [Modern Portfolio Theory](https://www.investopedia.com/terms/m/modernportfoliotheory.asp).

### What is a Select Sector SPDR ETF?

Each GICS sector has a matching **[Select Sector SPDR](https://www.ssga.com/us/en/institutional/etfs/spy)** exchange-traded fund managed by State Street (XLK for Info Tech, XLF for Financials, XLV for Health Care, ...). These ETFs *are* the sector — their holdings are the closest thing to a canonical "list of stocks in this sector." See [Investopedia: SPDR](https://www.investopedia.com/terms/s/spdr.asp) for background, [Investopedia: ETF](https://www.investopedia.com/terms/e/etf.asp) for the ETF wrapper itself, and [Investopedia: Sector ETF](https://www.investopedia.com/terms/s/sector-etf.asp) for the specific vehicle we're piggybacking on here.

`obb.techtrade.segments()` returns the 11 GICS sectors. Each sector knows its benchmark ETF and how its universe is resolved — by default, the ETF's current holdings. This is the input space for the scan: today's setups will be sourced from these sectors.

In [8]:
segments = obb.techtrade.segments().results
print(f"Sectors available: {len(segments)}\n")
for s in segments:
    print(f"  {s.segment:<25} benchmark={s.benchmark_etf or '(none)':<6} source={s.universe_source} top_n={s.top_n}")

Sectors available: 11

  Information Technology    benchmark=XLK    source=etf_holdings top_n=10
  Financials                benchmark=XLF    source=etf_holdings top_n=10
  Energy                    benchmark=XLE    source=etf_holdings top_n=10
  Health Care               benchmark=XLV    source=etf_holdings top_n=10
  Consumer Discretionary    benchmark=XLY    source=etf_holdings top_n=10
  Consumer Staples          benchmark=XLP    source=etf_holdings top_n=10
  Industrials               benchmark=XLI    source=etf_holdings top_n=10
  Materials                 benchmark=XLB    source=etf_holdings top_n=10
  Real Estate               benchmark=XLRE   source=etf_holdings top_n=10
  Utilities                 benchmark=XLU    source=etf_holdings top_n=10
  Communication Services    benchmark=XLC    source=etf_holdings top_n=10


In [9]:
### Reading the output

Every sector has a `benchmark_etf` (the Select Sector SPDR — XLK, XLF, XLV, ...) and a `universe_source` (`etf_holdings` by default). When a `scan` runs, it expands each ETF into its **current** holdings and ranks them. Today's universe is therefore *not hardcoded* — it floats with the ETF's quarterly rebalances.

**Why this matters:** if a stock gets removed from XLK next quarter (index-committee decision or M&A), it silently drops from tomorrow's scan with no code change on Alex's side. Conversely, a new IPO added to XLK becomes eligible the day it's added. This is the intended behavior — the scanner tracks the ETF's opinion of what the sector currently is, rather than a stale list Alex maintains by hand.

If you'd rather pin the universe to a fixed snapshot for reproducibility (e.g., to backtest yesterday's picks without today's rebalance polluting the results), override the `universe_source` — see the README § Universes. This distinction between **live universe** and **snapshotted universe** is a classic source of [survivorship bias](https://www.investopedia.com/terms/s/survivorshipbias.asp) if you're not careful; be explicit about which one you're using.

SyntaxError: invalid character '—' (U+2014) (4051518442.py, line 3)

## 3. Sanity-check one sector

Before scanning all 11, Alex pokes one sector to see what the mover ranking looks like. `metric="pct_change"` ranks by today's price move; `top_n=10` keeps the top 10 movers within the sector.

### Why sanity-check first?

**Fail-fast, don't fail-slow.** The full scan across 11 sectors is the day's most expensive operation (1-15 minutes). If the plumbing is broken today — API rate-limited, sector universe empty, exchange holiday you forgot about, provider returning stale prices — Alex wants to know from a **3-second** query, not a 15-minute one. This mirrors the software-engineering principle of [smoke testing](https://en.wikipedia.org/wiki/Smoke_testing_(software)): cheap check before expensive check.

The specific metric `pct_change` (today's price % change vs the prior close) is chosen here because it's the most straightforward mover signal to eyeball — a healthy market has some names up 2-5%, some down 2-5%, and volume in the millions. If everything you see is ±0.05% with zero volume, something's wrong upstream and it's not worth running the full scan.

In [10]:
# ~3-10s on a warm cache; longer cold.
movers_it = obb.techtrade.movers(
    segment="Information Technology",
    metric="pct_change",
    top_n=10,
).results

# `movers_it` is a list[MoverList]; one entry for the segment, containing the ranked tickers.
ml = movers_it[0]
print(f"Segment: {ml.segment}  |  as_of: {ml.as_of}  |  movers: {len(ml.movers)}\n")
for m in ml.movers:
    print(f"  #{m.rank:<2} {m.symbol:<6} pct_change={m.pct_change:+.2f}%  volume={m.volume:,.0f}")

Cache analysis failed for IXTU6: No data found for IXTU6.
movers: OHLCV fetch failed for IXTU6: 
[Empty] -> No data found for IXTU6. — dropping from candidate pool


Segment: Information Technology  |  as_of: 2026-07-08  |  movers: 10

  #1  AKAM   pct_change=+9.50%  volume=4,184,664
  #2  ANET   pct_change=+7.31%  volume=8,213,499
  #3  AVGO   pct_change=+5.32%  volume=21,968,786
  #4  SMCI   pct_change=+5.01%  volume=26,549,847
  #5  NXPI   pct_change=+4.22%  volume=1,712,384
  #6  MPWR   pct_change=+4.08%  volume=439,915
  #7  CIEN   pct_change=+4.03%  volume=1,235,049
  #8  NVDA   pct_change=+3.83%  volume=104,083,916
  #9  SNDK   pct_change=+3.82%  volume=9,461,096
  #10 TXN    pct_change=+3.64%  volume=4,122,761


### What Alex looks for here

- **A reasonable spread of `pct_change`** — if every name is at ±0.1% the market is flat and today probably isn't a setup day. Trend-following systems (see §4 below) need directional movement to work; flat tape starves them of signal.
- **[Volume](https://www.investopedia.com/terms/v/volume.asp) should not be zero.** Volume is the number of shares traded. A top-mover with no volume is [**illiquid**](https://www.investopedia.com/terms/l/liquidity.asp) — Alex can't safely take a position because entering AND exiting the trade will move the price against him. This price impact is called [**slippage**](https://www.investopedia.com/terms/s/slippage.asp), and it's the silent killer of small-account traders. Rule of thumb: skip anything with average daily volume below ~500K shares if you're trading retail size.
- **No surprise names.** He should recognize most of the top-10 in IT (Apple / Microsoft / Nvidia / etc.). If a microcap he's never heard of shows up at #1 with a +40% move, that's a flag — often a [pump-and-dump](https://www.investopedia.com/terms/p/pumpanddump.asp), a rumor-driven spike, or a data-provider error. Novices lose money chasing these; disciplined traders let them go.

### Why "sanity check" > "trust the tool"

Even a correctly-working scanner reflects the market it's fed. A power outage at FMP, a corporate action mishandled upstream, or an exchange halt can produce a ranking that *looks* legitimate but is garbage. Two seconds of eyeballing catches most of these; automated tests catch none of them. This is the [**garbage-in-garbage-out**](https://en.wikipedia.org/wiki/Garbage_in,_garbage_out) principle applied to trading — the tool is only as good as the data it consumes, and only a human recognizes that today's data looks weird.

## 4. The morning scan

Now the headline command. `obb.techtrade.scan(...)` runs the full pipeline (movers → indicator panel → confluence signal → rule → sizing → orders → recommendation) across **all 11 sectors** and ranks the resulting plans by conviction.

### What is "confluence" and why does techtrade rank by it?

A **confluence signal** is when [multiple independent indicators agree](https://www.investopedia.com/terms/c/confluence.asp) on direction. Instead of trusting any one indicator (which will be wrong ~40-50% of the time on its own), the engine polls a *panel* of them and only recommends a trade when a **weighted majority** votes the same way. This is the same principle as [ensemble methods](https://en.wikipedia.org/wiki/Ensemble_learning) in machine learning: independent noisy signals combined intelligently produce a less noisy output.

The four indicator families techtrade uses:

| Family | Example indicators | What they measure |
|---|---|---|
| **[Trend](https://www.investopedia.com/terms/t/trendtrading.asp)** | [SMA / EMA](https://www.investopedia.com/terms/s/sma.asp), [ADX](https://www.investopedia.com/terms/a/adx.asp), Ichimoku | Direction and strength of the primary trend |
| **[Momentum](https://www.investopedia.com/terms/m/momentum.asp)** | [RSI](https://www.investopedia.com/terms/r/rsi.asp), [MACD](https://www.investopedia.com/terms/m/macd.asp), [Stochastic](https://www.investopedia.com/terms/s/stochasticoscillator.asp) | Rate of change / overbought-oversold |
| **[Volatility](https://www.investopedia.com/terms/v/volatility.asp)** | [ATR](https://www.investopedia.com/terms/a/atr.asp), [Bollinger Bands](https://www.investopedia.com/terms/b/bollingerbands.asp), Keltner | How much and how quickly price moves |
| **[Volume](https://www.investopedia.com/terms/v/volume.asp)** | [OBV](https://www.investopedia.com/terms/o/onbalancevolume.asp), [CMF](https://www.investopedia.com/terms/c/chaikinmoneyflow.asp), [MFI](https://www.investopedia.com/terms/m/mfi.asp) | Conviction behind the price move |

Each **preset** (`trend_follow`, `mean_revert`, `breakout`) assigns different **weights** to these families — that's how the same underlying signals produce different recommendations depending on the trading style. §9 below shows all three side by side.

### Argument choices below — Alex's morning defaults

| Arg | Value | Why |
|---|---|---|
| `metric="pct_change"` | Rank movers within each sector by today's % move | Standard "what's moving" filter |
| `top_n=5` | Keep top 5 movers per sector (= up to 55 candidates) | Bounds wall-clock; 5 per sector is plenty of diversity |
| `preset="trend_follow"` | Confluence weights tilted to trend signals | Default; §9 below demos `mean_revert` and `breakout` |
| `risk=0.01` | 1% of notional risked per trade | The [**Van Tharp 1% rule**](https://www.investopedia.com/articles/trading/09/risk-management.asp) — no single loss erases more than 1% of the account. Even a 10-loss losing streak leaves ~90% intact. §10 below shows how to change this. |

**A note on wall-clock:** cold-cache scans hit `fmp_cached` for every symbol in every sector. Warm-cache is roughly 100× faster because responses are memoized on disk. If this is your first run today, expect 5-15 minutes; subsequent runs the same day complete in under a minute.

In [11]:
# Wall-clock: 1-3 minutes warm; 5-15 minutes cold. This is the morning's heaviest call.
scan_result = obb.techtrade.scan(
    metric="pct_change",
    top_n=5,
    preset="trend_follow",
    risk=0.01,
)
plans = scan_result.results
print(f"scan returned {len(plans)} plans across {len({p.segment for p in plans})} sectors.")

Cache analysis failed for IXTU6: No data found for IXTU6.
movers: OHLCV fetch failed for IXTU6: 
[Empty] -> No data found for IXTU6. — dropping from candidate pool
Cache analysis failed for BRK.B: Unauthorized FMP request -> 402 -> Premium Query Parameter: 'Special Endpoint : This value set for 'symbol' is not available under your current subscription please visit our subscription page to upgrade your plan at https://financialmodelingprep.com/
movers: OHLCV fetch failed for BRK.B: 
[Error] -> Unauthorized FMP request -> 402 -> Premium Query Parameter: 'Special Endpoint : This value set for 'symbol' is not available under your current subscription please visit our subscription page to upgrade your plan at https://financialmodelingprep.com/ — dropping from candidate pool
Cache analysis failed for IXAU6: No data found for IXAU6.
movers: OHLCV fetch failed for IXAU6: 
[Empty] -> No data found for IXAU6. — dropping from candidate pool
Cache analysis failed for IXPU6: No data found for IXPU6

scan returned 20 plans across 10 sectors.


### If you got 0 plans

Not a bug — it means every candidate today scored below the entry threshold (default `|score| >= 0.4`). Real markets have flat, choppy, or news-frozen days where no confluence signal is strong enough to trade. Options:

- **Try a different `preset`** — `mean_revert` will flag pullbacks the trend-follower ignored; `breakout` catches volatility-led setups.
- **Lower the entry threshold** via the `plan` command directly (advanced — see notebook 03).
- **Wait a day.** "No setup" is a legitimate outcome of discipline. Elder puts it bluntly in [*Trading for a Living*](https://en.wikipedia.org/wiki/Alexander_Elder): *"There is nothing wrong with sitting on cash. The market will still be there tomorrow."*

Over a year, a systematic trader typically finds actionable setups on **maybe 60-80% of trading days**. Empty-scan days are part of the distribution, not evidence the tool is broken.

## 5. Quick-look ranking

Before the Excel export, Alex glances at the top of the list. Conviction is bucketed as `High`/`Medium`/`Low` from the absolute confluence score (the README § Commands describes the bucketing).

### Why "conviction bucketing"?

A raw score is a number — humans reason poorly about "0.63 vs 0.58." Bucketing collapses the score into 3 labels a trader can actually act on: **High = take it**, **Medium = maybe take a smaller size**, **Low = skip**. This kind of coarse categorization mirrors how professional trading desks translate quant-model output into desk-actionable "conviction levels" (see [Investopedia: Quantitative Trading](https://www.investopedia.com/terms/q/quantitative-trading.asp) for the broader context).

The thresholds are set so that historically ~5-15% of scanned candidates land in `High` on a normal day (the exact rate varies with market regime and preset). If your High count is unusually large or small today, that's information about the tape, not a bug — an unusually High-count day often means the market has strong directional consensus, an unusually Low-count day usually means it's in transition.

In [12]:
import pandas as pd

rows = [
    {
        "symbol": p.symbol,
        "segment": p.segment[:20],
        "score": p.signal.score,
        "action": p.recommendation.action,
        "conviction": p.recommendation.conviction,
        "entry": float(p.recommendation.entry_price),
        "stop": float(p.recommendation.stop_price),
        "target": float(p.recommendation.target_price),
        "r:r": p.recommendation.risk_reward,
        "qty": float(p.recommendation.position_size),
    }
    for p in plans
]

df = pd.DataFrame(rows)
if df.empty:
    print("No plans today; nothing to rank.")
else:
    # Show the top 15 by absolute score.
    df_sorted = df.reindex(df["score"].abs().sort_values(ascending=False).index)
    df_sorted.head(15).reset_index(drop=True)

### Reading the table

- `score` is in `[-1, +1]`. **Sign** = direction (positive = long, negative = short). **Magnitude** = confluence strength.
- `action` is the bucketed call: `BUY` / `SELL_SHORT` / `HOLD/FLAT`. (For a primer on the mechanics of shorting see [Investopedia: Short Selling](https://www.investopedia.com/terms/s/shortselling.asp).)
- `conviction` thresholds (from PRD §12.2): `High` if `|score| >= 0.7`, `Medium` if `|score| >= 0.4`, else `Low` (filtered out by the entry threshold).
- **`r:r`** is the [**reward-to-risk ratio**](https://www.investopedia.com/terms/r/riskrewardratio.asp) at the stop/target levels. A `r:r` of 2.0 means the target is twice as far from entry as the stop.
- `qty` is the position size at the 1% risk Alex specified. It scales **inversely** with the stop distance: **tight stop = bigger position; wide stop = smaller position** — always sized so a stopped-out trade loses exactly 1% of the account. See [Investopedia: Position Sizing](https://www.investopedia.com/terms/p/positionsizing.asp) for the general concept.

### Why does r:r matter mathematically?

A trader's [**expectancy per trade**](https://www.investopedia.com/articles/trading/06/riskrewardexpectancy.asp) is:

```
expectancy = (win_rate × avg_win) − (loss_rate × avg_loss)
```

If `r:r = 1.0` (win the same amount as you lose), you need win_rate > 50% just to break even after commissions. If `r:r = 2.0`, break-even is ~33% win rate. If `r:r = 3.0`, ~25%. **This is why professional traders talk about r:r before win rate** — a modest edge with good r:r beats a strong edge with bad r:r almost every time.

| r:r | Break-even win rate | Interpretation |
|---:|---:|---|
| 1.0 | 50% | Coin-flip; commissions kill you |
| 1.5 | 40% | Modest edge suffices |
| **2.0** | **33%** | **Alex's floor — even a losing streak leaves positive expectancy** |
| 3.0 | 25% | Trend-following territory; long tails |
| 5.0 | 17% | Suspicious — target probably unrealistic (§11 checklist) |

Alex's filter of `r:r >= 2.0` in §6 below is a hard floor derived from this math: even if the confluence engine is only right 40% of the time, a `r:r >= 2.0` portfolio has positive expectancy.

## 6. Filter to today's actionable list

Alex doesn't take every signal. He filters to:

1. **`High` conviction only** (drops the medium-noise tier).
2. **`r:r >= 2.0`** (drops trades with bad reward-to-risk — see §5's expectancy math).
3. **`segment` diversity** — cap at 2 plans per sector so he isn't all-in on one sector's idiosyncrasies.

### Why the diversity cap?

If Alex takes all 5 High-conviction picks from Information Technology on a day that IT collapses, he's taken **one bet 5 times** — the trades are correlated at ~0.9 because tech stocks move together on tech-sector news. The [**correlation-versus-diversification tradeoff**](https://www.investopedia.com/terms/d/diversification.asp) is why professional risk desks impose sector caps at the portfolio level; Alex just applies the same principle at his morning-scan level.

Rule of thumb (empirically from Harry Markowitz's [Modern Portfolio Theory](https://www.investopedia.com/terms/m/modernportfoliotheory.asp), which won the 1990 Nobel Prize in Economics): **a portfolio isn't diversified until each position has correlation < 0.5 with every other**. Two picks per sector across 6+ sectors comfortably clears that bar; five picks in one sector does not. For a deeper treatment see [Investopedia: Correlation and Portfolio Performance](https://www.investopedia.com/articles/financial-theory/09/uncorrelated-assets-diversification.asp).

These three filters are Alex's **standing rules** — his personal risk policy. They're not techtrade defaults. Edit this cell to match your own discipline (some traders cap at 1 per sector; some allow 3; some add filters on absolute stop distance or absolute conviction score). The key is: **write the rules down, then apply them mechanically**. Discretionary "I'll skip this one because it feels wrong" is what discipline is supposed to prevent.

In [23]:
from collections import defaultdict

import numpy as np


def max_hold_date(anchor, bars):
    """Latest exit date = anchor + `bars` trading days.

    Uses a business-day offset (Mon-Fri). This is an approximation: it skips
    weekends but NOT market holidays, so the true latest-exit date can be a few
    days later than shown. `bars=None` means the time-stop is disabled.
    """
    if bars is None:
        return None
    return (
        np.busday_offset(np.datetime64(anchor, "D"), int(bars), roll="forward")
        .astype("datetime64[D]")
        .astype(str)
    )


actionable: list = []
per_sector = defaultdict(int)
PER_SECTOR_CAP = 2

# Sort by absolute score descending so the strongest in each sector wins the cap.
for p in sorted(plans, key=lambda p: -abs(p.signal.score)):
    rec = p.recommendation
    if rec.conviction != "High":
        continue
    if rec.risk_reward < 2.0:
        continue
    if per_sector[p.segment] >= PER_SECTOR_CAP:
        continue
    actionable.append(p)
    per_sector[p.segment] += 1

print(f"Actionable after filters: {len(actionable)} / {len(plans)} plans\n")
for p in actionable:
    rec = p.recommendation
    bars = rec.time_stop_bars
    by = max_hold_date(rec.as_of, bars)
    hold_str = f"hold<={bars}d by {by}" if by else "hold=no time-stop"
    print(
        f"  {p.symbol:<6} {p.segment[:18]:<18} score={p.signal.score:+.3f} "
        f"{rec.action:<10} entry=${float(rec.entry_price):>10,.2f}  "
        f"stop=${float(rec.stop_price):>10,.2f}  target=${float(rec.target_price):>10,.2f}  "
        f"r:r={rec.risk_reward:.1f}  qty={rec.position_size}  {hold_str}"
    )


Actionable after filters: 7 / 20 plans

  AJG    Financials         score=+0.978 BUY        entry=$    256.46  stop=$    242.55  target=$    284.29  r:r=2.0  qty=71  hold<=20d by 2026-08-05
  SCHW   Financials         score=+0.978 BUY        entry=$    102.50  stop=$     97.62  target=$    112.26  r:r=2.0  qty=204  hold<=20d by 2026-08-05
  MPC    Energy             score=+0.930 BUY        entry=$    277.44  stop=$    260.06  target=$    312.22  r:r=2.0  qty=57  hold<=20d by 2026-08-05
  VLO    Energy             score=+0.904 BUY        entry=$    278.82  stop=$    258.81  target=$    318.83  r:r=2.0  qty=49  hold<=20d by 2026-08-05
  ADM    Consumer Staples   score=+0.810 BUY        entry=$     80.13  stop=$     75.93  target=$     88.53  r:r=2.0  qty=238  hold<=20d by 2026-08-05
  D      Utilities          score=+0.717 BUY        entry=$     69.90  stop=$     67.43  target=$     74.85  r:r=2.0  qty=404  hold<=20d by 2026-08-05
  ANET   Information Techno score=+0.708 BUY        entry

## 7. Why does each setup look good? The audit trail

techtrade's competitive advantage over a black-box scanner is that **every recommendation comes with its full vote attribution**. The `signal.votes` field shows which indicator in which family voted which direction with which weight. Alex can always answer "why long?"

### Why explainability matters (not just for one-off curiosity)

Two reasons this matters more than most novices realize:

1. **You cannot improve what you cannot inspect.** If a scanner is a black box, you can't tell whether a losing streak came from bad luck (variance) or a broken indicator (bug in the code or a regime change that has made one family unreliable). Attributed signals let you say *"I lost 3 in a row and all 3 losing votes came from the MACD momentum family — momentum is misbehaving in this regime, so I'll temporarily downweight it."* This is [**model diagnostics**](https://en.wikipedia.org/wiki/Regression_diagnostic) applied to trading.
2. **You cannot trust what you cannot audit.** When Alex is tempted to override the engine (*"I have a feeling about this one"*) the audit trail is what talks him down: *"the engine says short with weight 0.85 concentrated in trend + momentum; my feeling is one data point vs eight independent indicators."* This is [**the discipline of not deviating from the plan**](https://www.investopedia.com/articles/trading/09/trading-mistakes.asp), which Alexander Elder rates as the single largest edge a retail trader can develop. See also [Investopedia: Behavioral Finance](https://www.investopedia.com/terms/b/behavioralfinance.asp) — the field that studies the systematic ways human psychology sabotages trading decisions.

Novice-to-expert progression: for the first month, read the audit for every actionable plan. After that, spot-check when a plan surprises you. This is how you learn what the confluence engine is actually measuring — and where its blind spots are.

In [20]:
if not actionable:
    print("No actionable plans today; nothing to audit.")
else:
    p = actionable[0]
    print(f"--- Audit for {p.symbol} ({p.segment}) ---")
    print(f"Direction: {p.signal.direction}  |  Composite score: {p.signal.score:+.3f}\n")
    print(f"{'family':<12} {'name':<14} {'vote':>6} {'weight':>8} {'contrib':>10}")
    for v in sorted(p.signal.votes, key=lambda v: -abs(v.vote * v.weight)):
        contrib = v.vote * v.weight
        print(f"{v.family:<12} {v.name:<14} {v.vote:+.2f}   {v.weight:.2f}     {contrib:+.4f}")
    print(f"\nReasoning:  {p.recommendation.reasoning}")
    print(f"Caveats:    {p.recommendation.caveats}")

--- Audit for AJG (Financials) ---
Direction: long  |  Composite score: +0.978

family       name             vote   weight    contrib
trend        macd_hist      +1.00   0.40     +0.4000
trend        ema_cross      +1.00   0.40     +0.4000
momentum     rsi            +1.00   0.25     +0.2500
momentum     stoch          +1.00   0.25     +0.2500
volatility   bb_pctb        +1.00   0.20     +0.2000
volume       obv_slope      +1.00   0.15     +0.1500
volume       cmf            +1.00   0.15     +0.1500

Reasoning:  Long AJG: trend strongly positive (macd_hist+, ema_cross+), momentum strongly confirming (rsi+, stoch+), volatility breakout (bb_pctb+), and volume confirming (obv_slope+, cmf+). Stop 2xATR below entry (-5.4%); target at 2.0R (+10.8%).
Caveats:    No paper fill - levels are planned, not realized.


### How to read the audit

- `vote` is the indicator's directional reading on `[-1, +1]`.
- `weight` is what the preset assigns to that family (`trend_follow` defaults: trend 0.40 / momentum 0.25 / volatility 0.20 / volume 0.15).
- `contrib` is `vote × weight` — the indicator's **signed contribution** to the composite score.
- The **`reasoning`** field is a **deterministic** natural-language summary built from the top contributors. **No LLM** — if the same panel + weights are fed again, the same reasoning text comes out. This is intentional: reproducibility is worth more than prose quality for an audit trail. (Contrast with generative-AI narrative tools that produce different rationalizations for identical inputs — those are useless as evidence in a post-trade review.)
- The **`caveats`** field flags risk concerns the engine noticed (low volume, wide stop, opposing high-weight indicator, upcoming earnings, etc.). Read them; they're the engine's way of saying *"yes, but…"*.

### The indicator families, expanded

If any of these terms are unfamiliar, click through for a solid single-page primer:

- **[Trend](https://www.investopedia.com/terms/t/trendtrading.asp)** — [SMA/EMA](https://www.investopedia.com/terms/s/sma.asp), [ADX](https://www.investopedia.com/terms/a/adx.asp), Ichimoku. *Question answered:* which way is price going?
- **[Momentum](https://www.investopedia.com/terms/m/momentum.asp)** — [RSI](https://www.investopedia.com/terms/r/rsi.asp), [MACD](https://www.investopedia.com/terms/m/macd.asp), [Stochastic](https://www.investopedia.com/terms/s/stochasticoscillator.asp). *Question answered:* how fast is it going, and is it overextended?
- **[Volatility](https://www.investopedia.com/terms/v/volatility.asp)** — [ATR](https://www.investopedia.com/terms/a/atr.asp), [Bollinger Bands](https://www.investopedia.com/terms/b/bollingerbands.asp), Keltner. *Question answered:* how much range should we expect, and are we compressed or expanded?
- **[Volume](https://www.investopedia.com/terms/v/volume.asp)** — [OBV](https://www.investopedia.com/terms/o/onbalancevolume.asp), [CMF](https://www.investopedia.com/terms/c/chaikinmoneyflow.asp), [MFI](https://www.investopedia.com/terms/m/mfi.asp). *Question answered:* is real money behind the move, or is it thin?

The classical reference for these indicators (definitions, formulas, standard periods) is Murphy's *[Technical Analysis of the Financial Markets](https://www.investopedia.com/articles/active-trading/010615/top-technical-analysis-books.asp)* (NYIF, 1999) — the industry-standard textbook.

## 8. Export to Excel (the morning artifact)

Alex's final morning step is the 6-sheet Excel workbook. Each sheet is the same data sliced differently:

| Sheet | What's on it |
|---|---|
| **Recommendations** | One row per plan with the action/conviction/levels (with the "research/paper — not investment advice" disclaimer banner) |
| **Levels** | Entry / stop / target / stop distance % / target distance % / [ATR](https://www.investopedia.com/terms/a/atr.asp) per plan |
| **Reasoning** | The full `reasoning` text + `top_factors` per plan (the audit trail from §7) |
| **Orders** | The [broker-ready order legs](https://www.investopedia.com/articles/investing/091614/basics-trading-system.asp) (entry + stop + target + time-exit) per plan |
| **Fills** | Paper fills if `scan(simulate=True)` returned them (the v1 router does not yet — see notebook 03) |
| **Summary** | One-row dashboard: total plans, score distribution, sector breakdown |

Default output path is `Analysis/exports/techtrade_<date>.xlsx`. Conditional formatting on the Recommendations sheet color-codes action and risk-reward.

### Why paper-first, then real money?

Before wiring an entry order into a live broker account, a plan should be [**paper-traded**](https://www.investopedia.com/terms/p/papertrade.asp) — simulated as if it were real, tracked to closure, and only "promoted" to real money after you've watched it work through **several dozen actual market days**. This is how professional prop desks onboard traders and is the industry-standard [risk-management](https://www.investopedia.com/terms/r/riskmanagement.asp) safeguard against a plausible-looking model that's actually [**overfit**](https://www.investopedia.com/terms/o/overfitting.asp) to recent history.

Notebook 03 walks through the paper-fill mechanics; notebook 04 does the [walk-forward validation](https://www.investopedia.com/terms/w/walkforward.asp) that either promotes or demotes a strategy. If those terms sound intimidating, the point is exactly that they *should* — anyone who tells you a trading system is "ready to deploy" without walk-forward validation is either naive or selling something.

### Reading the workbook

The Excel file is designed to be **printable and reviewable offline** — Alex can open it on his phone during coffee, mark up the plans he'll take, and then execute against his broker later. This paper-first workflow is why the workbook is the morning's key deliverable, not a live dashboard.

In [21]:
# Export ONLY the actionable plans, not every signal that came back.
# Wall-clock: 5-30s depending on plan count + I/O.
export_result = obb.techtrade.export(
    plans=actionable,
    path=None,  # use the default `Analysis/exports/techtrade_<date>.xlsx`
    engine="openpyxl",  # ships with the bare install; `xlsxwriter` is the optional alternative
)
xlsx_path = Path(export_result.results)
print(f"Workbook written to: {xlsx_path}")
print(f"Size: {xlsx_path.stat().st_size / 1024:.1f} KB")

Workbook written to: H:\masterswork\git\OpenBBTechnical\Analysis\exports\techtrade_2026-07-08.xlsx
Size: 13.4 KB


In [ ]:
# Peek at the sheet names + the first few rows of Recommendations so Alex sees
# what landed without opening Excel.
import openpyxl

wb = openpyxl.load_workbook(xlsx_path, read_only=True, data_only=True)
print(f"Sheets ({len(wb.sheetnames)}): {wb.sheetnames}\n")

rec = wb["Recommendations"]
print("--- Recommendations sheet (first 4 rows) ---")
for i, row in enumerate(rec.iter_rows(values_only=True, max_row=4)):
    cells = [str(c)[:18] if c is not None else "" for c in row[:8]]
    print("  " + " | ".join(cells))
wb.close()

## 9. Variation: try the other two presets

The same scan with a different preset gives a different list. Each preset embodies a distinct **trading philosophy** — a different theory of *how markets work*:

| Preset | Philosophy | Origin |
|---|---|---|
| `trend_follow` | Prices trend; ride established moves | The [**Turtle Traders**](https://en.wikipedia.org/wiki/Turtle_Traders) experiment (Richard Dennis, 1983); [Ed Seykota](https://en.wikipedia.org/wiki/Ed_Seykota). See [Investopedia: Trend Trading](https://www.investopedia.com/terms/t/trendtrading.asp). |
| `mean_revert` | Prices oscillate around fair value; fade extremes | Statistical arbitrage; [Renaissance Technologies](https://en.wikipedia.org/wiki/Renaissance_Technologies) (Jim Simons). See [Investopedia: Mean Reversion](https://www.investopedia.com/terms/m/meanreversion.asp). |
| `breakout` | Volatility clusters; buy new highs when volume confirms | [Donchian Channel](https://www.investopedia.com/terms/d/donchianchannels.asp) (Richard Donchian, ~1960); [William O'Neil's CAN SLIM](https://www.investopedia.com/terms/c/canslim.asp). |

**All three philosophies are empirically supported** in the academic literature — none is universally "best":

- Trend / momentum: [Asness, Moskowitz & Pedersen, "Value and Momentum Everywhere" (2013, *Journal of Finance*)](https://onlinelibrary.wiley.com/doi/10.1111/jofi.12021).
- Mean reversion: [Poterba & Summers, "Mean Reversion in Stock Prices" (1988, *Journal of Financial Economics*)](https://www.nber.org/papers/w2343).
- Breakout with volume confirmation: [Kaminski & Lo, "When Do Stop-Loss Rules Stop Losses?" (2014, *Journal of Financial Markets*)](https://www.sciencedirect.com/science/article/abs/pii/S1386418113000724).

They take **turns leading** depending on market regime: trend follows dominates in strongly-trending environments; mean-reversion dominates in range-bound tape; breakout dominates when volatility is compressed and about to expand. Multi-preset scanning is Alex's way of not committing to a philosophy without evidence.

Alex sometimes runs all three and looks at the **overlap** — a name that shows up under multiple presets is a stronger signal than a name picked by just one, because it means multiple independent theories all say the same thing.

**Heads-up:** each preset takes another full scan cycle. Two more presets = another 2-6 minutes warm-cache.

In [ ]:
# Re-scan with mean_revert and breakout; compare the High-conviction symbols.
preset_picks = {"trend_follow": {p.symbol for p in actionable}}

for preset in ("mean_revert", "breakout"):
    res = obb.techtrade.scan(metric="pct_change", top_n=5, preset=preset, risk=0.01).results
    high = {p.symbol for p in res if p.recommendation.conviction == "High"}
    preset_picks[preset] = high
    print(f"{preset:<14} High-conviction picks: {sorted(high)}")

# Overlap analysis.
all_picks = set().union(*preset_picks.values())
for sym in sorted(all_picks):
    by_preset = [name for name, s in preset_picks.items() if sym in s]
    print(f"  {sym:<6} flagged by {len(by_preset)}/3 presets: {by_preset}")

### When to trust the overlap

- **3/3 overlap**: rare, strong cross-preset agreement. The name is moving in a way that satisfies trend-followers AND mean-reverters AND breakout traders all at once. Worth a deeper look in notebook 03.
- **2/3 overlap**: common; usually means a trending name that's also pulling back (trend + mean-revert agree) or breaking out (trend + breakout agree). Decent signal.
- **1/3 overlap**: the preset matters. Treat the pick as preset-specific — a `mean_revert`-only flag means Alex needs to be in a fade-the-rally mood that day.

## 10. Variation: change the risk knob

`risk=0.01` is conventional. `risk=0.005` is conservative (half the position size, smaller losses on a stopped trade); `risk=0.02` is aggressive (twice the size, twice the loss). The `qty` column in the table scales linearly with risk.

### The 1% rule — where does it come from?

Van Tharp popularized the [**1% risk rule**](https://www.investopedia.com/articles/trading/09/risk-management.asp) in *[Trade Your Way to Financial Freedom](https://vantharp.com/trade-your-way-to-financial-freedom/)* (McGraw-Hill, 1998). The math behind it — the [**risk of ruin**](https://en.wikipedia.org/wiki/Risk_of_ruin) equation — says that if you risk 1% per trade with a modestly-positive edge (win rate ~50%, `r:r ≈ 1.5`), the probability of drawing down your account by 50% before doubling it is roughly **1 in 100**. Raise per-trade risk to 5% and that same probability jumps to over 50% — you're now more likely to blow up than double.

The related [**Kelly criterion**](https://www.investopedia.com/articles/trading/04/091504.asp) (Kelly, 1956) computes the "mathematically optimal" bet size given win rate and payoff. In practice, most professional traders use **half-Kelly** or less because:

1. Kelly maximizes long-run growth but tolerates brutal drawdowns.
2. Real win rate and payoff are estimates with wide error bars; sizing to full Kelly on a wrong estimate is ruinous.
3. Human psychology cannot execute Kelly-sized positions after a losing streak — you'll cut size at exactly the wrong moment.

**1% is essentially "quarter-Kelly for a typical retail edge"** — small enough to survive a bad estimate and a run of bad luck, large enough to compound meaningfully over years. For a deeper treatment of position-sizing theory, see [Investopedia: Position Sizing](https://www.investopedia.com/terms/p/positionsizing.asp) and the classic Ralph Vince book [*The Handbook of Portfolio Mathematics*](https://www.wiley.com/en-us/The+Handbook+of+Portfolio+Mathematics%3A+Formulas+for+Optimal+Allocation+and+Leverage-p-9780471757689) (Wiley, 2007).

### Sizing math, worked out

The formula the engine uses is:

```
qty = round( (account_equity × risk) / (entry_price - stop_price) )
```

So if Alex has a $100,000 account and risks 1%, he risks $1,000 per trade. If entry is $150 and stop is $147.50 (a $2.50 stop distance), the engine sizes `qty = 1000 / 2.50 = 400` shares. If instead the stop is $145 (a $5 stop distance), `qty = 1000 / 5 = 200` shares. **The dollar loss on the stop is always $1,000** — that's the invariant the 1% rule enforces.

### Alex's rule of thumb

**Never run the morning scan with a `risk` value he hasn't thought through.** It's the single fastest way to blow up an account. If in doubt, halve it. You cannot get rich fast without also going broke fast; the math is symmetric. See [Investopedia: Drawdown](https://www.investopedia.com/terms/d/drawdown.asp) for what happens when you don't respect this — a 50% drawdown requires a 100% subsequent gain just to recover.

In [ ]:
# Show how qty scales for the top actionable plan at three risk levels.
# Re-running scan() at each risk is expensive; instead, we observe that the engine
# computes qty = round(risk * notional / (entry - stop)). Linear scaling = quick demo.
if actionable:
    p = actionable[0]
    rec = p.recommendation
    base_risk = 0.01
    base_qty = float(rec.position_size)
    print(f"{p.symbol} at default risk {base_risk:.1%}: qty = {base_qty}")
    for r in (0.005, 0.015, 0.02):
        scaled = base_qty * (r / base_risk)
        print(f"  same plan at risk {r:.1%}: qty = {scaled:.0f}")

## 11. Wrap-up checklist

Before the bell rings, Alex's discipline says verify these five things from the morning scan:

- [ ] **The workbook opened cleanly in Excel** — no missing sheets, the disclaimer is visible on the Recommendations sheet.
- [ ] **At least one filter narrowed the list.** If `len(actionable) == len(plans)` then his filters are no-ops; he should tighten them before relying on the output.
- [ ] **Reasoning text is non-empty** for every actionable plan. A blank `reasoning` means the audit trail didn't produce a narrative; treat that plan as suspect.
- [ ] **No top-3 symbol comes from a sector he wouldn't trade today.** (Energy on FOMC day, biotech on FDA-meeting day, retail on a big-box earnings day, etc. — see the [Investing.com economic calendar](https://www.investing.com/economic-calendar/) for scheduled market-moving events.)
- [ ] **R:R passes a gut check.** A `r:r = 5.0` looks great but probably means an unrealistically far target — check whether the target price sits above a well-known resistance level that's likely to reject price first.

If any of those fail, **re-scan with different args** or **skip the day**. **"Don't trade" is a valid — often the correct — output of a disciplined morning routine.**

Alexander Elder ([*Trading for a Living*](https://en.wikipedia.org/wiki/Alexander_Elder), 1993, Chapter 3): *the amateur trader looks for a reason to trade; the professional looks for a reason **not** to.*

---

## Recommended reading — deepen the concepts

**Free, high signal-to-noise:**
- [Investopedia: Trading for Beginners](https://www.investopedia.com/trading-4427756) — the best free primer.
- [SEC investor.gov](https://www.investor.gov/) — legally-mandated disclosures, common scams, beginner risk education.
- [FINRA Investor Education](https://www.finra.org/investors/learn-to-invest) — regulator-published guides on brokers, order types, and common pitfalls.
- [CFA Institute Research Foundation — free monographs](https://rpc.cfainstitute.org/en/research/foundation) — surprisingly readable primers on portfolio theory, factor investing, and risk management.

**Books (in order of "read this first"):**
- Van Tharp, *[Trade Your Way to Financial Freedom](https://vantharp.com/trade-your-way-to-financial-freedom/)* (McGraw-Hill, 1998) — position sizing, expectancy, risk of ruin.
- Alexander Elder, *[Trading for a Living](https://en.wikipedia.org/wiki/Alexander_Elder)* (Wiley, 1993) — the psychology of systematic trading.
- John Murphy, *[Technical Analysis of the Financial Markets](https://www.investopedia.com/articles/active-trading/010615/top-technical-analysis-books.asp)* (NYIF, 1999) — the industry-standard indicator reference.
- Michael Covel, *[The Complete TurtleTrader](https://www.michaelcovel.com/complete-turtletrader/)* (HarperCollins, 2007) — trend-following history and philosophy.

## What's next in the series

- **Notebook 03 — Single Position Deep Dive**: take ONE ticker from today's actionable list, compute its individual `plan`, materialize the `orders` legs, and paper-fill them forward with `simulate`. This is where Alex sees the no-look-ahead fill discipline up close.
- **Notebook 04 — The Validation Gate**: take ONE plan and run `obb.techtrade.validate(plan, method="wfo")`. [Walk-forward folds](https://www.investopedia.com/terms/w/walkforward.asp) + PBO + DSR + a verdict. The anti-[overfitting](https://www.investopedia.com/terms/o/overfitting.asp) gate that separates a genuine edge from data-mining artifacts.
- **Notebook 05 — Per-Sector Tuning**: `obb.techtrade.tune(segment)` with the `[tuneta]` extra. The optional path that proposes better indicator periods per sector.
- **Notebook 06 — Audit and Replay**: load yesterday's xlsx, replay what Alex actually did vs what the engine suggested, write a journal entry. Trading journals are the #1 skill-building tool no one uses — [see Investopedia on trading journals](https://www.investopedia.com/articles/trading/09/trading-journal.asp).

---

*End of notebook 02. Series: "A Developer Guide to Disciplined Trading". Maintained on the `trading_technicals` branch of `prajoria/OpenBB`.*